# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Veduisback/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("INSTALL azure")
con.execute("LOAD azure")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB + Hugging Face ready.")

HF token loaded: True
DuckDB + Hugging Face ready.


In [9]:
DIM_CONTENT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

print("dim_content schema:")

display(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT_PATH}')"
    ).df()
)

dim_content schema:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [10]:
q_dates = f"""
SELECT
    MIN(content_updated_date) AS min_updated_date,
    MAX(content_updated_date) AS max_updated_date,
    MIN(content_created_date) AS min_created_date,
    MAX(content_created_date) AS max_created_date,
    MIN(last_optimized_date) AS min_optimized_date,
    MAX(last_optimized_date) AS max_optimized_date
FROM read_parquet('{DIM_CONTENT_PATH}')
"""

display(con.execute(q_dates).df())

,min_updated_date,max_updated_date,min_created_date,max_created_date,min_optimized_date,max_optimized_date
0,2024-10-28,2026-07-06,2024-10-16,2026-07-06,2026-04-24,2026-07-06


In [11]:
DECISION_DATE = "2026-02-28"

q_staleness = f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        DATE_DIFF(
            'day',
            content_updated_date,
            DATE '{DECISION_DATE}'
        ) AS days_since_update
    FROM read_parquet('{DIM_CONTENT_PATH}')
    WHERE content_updated_date IS NOT NULL
),

bucketed AS (
    SELECT
        CASE
            WHEN days_since_update < 0 THEN 'Updated after cutoff'
            WHEN days_since_update <= 30 THEN '0-30 days'
            WHEN days_since_update <= 90 THEN '31-90 days'
            WHEN days_since_update <= 180 THEN '91-180 days'
            WHEN days_since_update <= 365 THEN '181-365 days'
            ELSE '366+ days'
        END AS staleness_bucket,
        COUNT(*) AS n
    FROM content
    GROUP BY 1
)

SELECT *
FROM bucketed
ORDER BY
    CASE staleness_bucket
        WHEN 'Updated after cutoff' THEN 0
        WHEN '0-30 days' THEN 1
        WHEN '31-90 days' THEN 2
        WHEN '91-180 days' THEN 3
        WHEN '181-365 days' THEN 4
        WHEN '366+ days' THEN 5
    END
"""

staleness_buckets = con.execute(q_staleness).df()
display(staleness_buckets)

,staleness_bucket,n
0,Updated after cutoff,383714
1,0-30 days,34076
2,31-90 days,1201
3,91-180 days,18239
4,181-365 days,14403
5,366+ days,67973


In [12]:
q_volume = f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        search_volume
    FROM read_parquet('{DIM_CONTENT_PATH}')
    WHERE search_volume IS NOT NULL
),

bucketed AS (
    SELECT
        CASE
            WHEN search_volume = 0 THEN '0'
            WHEN search_volume <= 100 THEN '1-100'
            WHEN search_volume <= 500 THEN '101-500'
            WHEN search_volume <= 1000 THEN '501-1000'
            WHEN search_volume <= 5000 THEN '1001-5000'
            ELSE '5001+'
        END AS volume_bucket,
        COUNT(*) AS n
    FROM content
    GROUP BY 1
)

SELECT *
FROM bucketed
ORDER BY
    CASE volume_bucket
        WHEN '0' THEN 0
        WHEN '1-100' THEN 1
        WHEN '101-500' THEN 2
        WHEN '501-1000' THEN 3
        WHEN '1001-5000' THEN 4
        WHEN '5001+' THEN 5
    END
"""

volume_buckets = con.execute(q_volume).df()
display(volume_buckets)

,volume_bucket,n
0,0,163631
1,1-100,174178
2,101-500,24316
3,501-1000,5890
4,1001-5000,6274
5,5001+,2695


In [13]:
FEB_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

q_staleness_audit = f"""
WITH performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d
    FROM read_parquet('{FEB_PATH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_DIFF(
            'day',
            content_updated_date,
            DATE '{DECISION_DATE}'
        ) AS days_since_update
    FROM read_parquet('{DIM_CONTENT_PATH}')
    WHERE content_updated_date IS NOT NULL
),

bucketed AS (
    SELECT
        c.client_hash_id,
        c.content_hash_id,
        CASE
            WHEN c.days_since_update < 0 THEN 'Updated after cutoff'
            WHEN c.days_since_update <= 30 THEN '0-30 days'
            WHEN c.days_since_update <= 90 THEN '31-90 days'
            WHEN c.days_since_update <= 180 THEN '91-180 days'
            WHEN c.days_since_update <= 365 THEN '181-365 days'
            ELSE '366+ days'
        END AS staleness_bucket
    FROM content c
)

SELECT
    b.staleness_bucket,
    COUNT(*) AS n,
    ROUND(AVG(p.impressions_30d), 1) AS avg_impressions_30d
FROM bucketed b
LEFT JOIN performance p
    ON b.client_hash_id = p.client_hash_id
    AND b.content_hash_id = p.content_hash_id
WHERE b.staleness_bucket != 'Updated after cutoff'
GROUP BY 1
ORDER BY
    CASE b.staleness_bucket
        WHEN '0-30 days' THEN 1
        WHEN '31-90 days' THEN 2
        WHEN '91-180 days' THEN 3
        WHEN '181-365 days' THEN 4
        WHEN '366+ days' THEN 5
    END
"""

staleness_audit = con.execute(q_staleness_audit).df()
display(staleness_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_impressions_30d
0,0-30 days,34076,796.5
1,31-90 days,1201,71.8
2,91-180 days,18239,106.9
3,181-365 days,14403,42.4
4,366+ days,67973,NaN


In [14]:
q_volume_audit = f"""
WITH performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d
    FROM read_parquet('{FEB_PATH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        search_volume
    FROM read_parquet('{DIM_CONTENT_PATH}')
    WHERE search_volume IS NOT NULL
),

bucketed AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CASE
            WHEN search_volume = 0 THEN '0'
            WHEN search_volume <= 100 THEN '1-100'
            WHEN search_volume <= 500 THEN '101-500'
            WHEN search_volume <= 1000 THEN '501-1000'
            WHEN search_volume <= 5000 THEN '1001-5000'
            ELSE '5001+'
        END AS volume_bucket
    FROM content
)

SELECT
    b.volume_bucket,
    COUNT(*) AS n,
    ROUND(AVG(p.impressions_30d), 1) AS avg_impressions_30d
FROM bucketed b
LEFT JOIN performance p
    ON b.client_hash_id = p.client_hash_id
    AND b.content_hash_id = p.content_hash_id
GROUP BY 1
ORDER BY
    CASE b.volume_bucket
        WHEN '0' THEN 0
        WHEN '1-100' THEN 1
        WHEN '101-500' THEN 2
        WHEN '501-1000' THEN 3
        WHEN '1001-5000' THEN 4
        WHEN '5001+' THEN 5
    END
"""

volume_audit = con.execute(q_volume_audit).df()
display(volume_audit)

,volume_bucket,n,avg_impressions_30d
0,0,163631,1305.2
1,1-100,174178,1303.5
2,101-500,24316,1432.4
3,501-1000,5890,1319.6
4,1001-5000,6274,1159.8
5,5001+,2695,859.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal checks

### Signal 1 — Staleness
**Verdict: CONFIRMED**

Staleness is directly linked to the refresh-oriented signal discussed in the session. At the 2026-02-28 decision cutoff, the measured February average impressions were 796.5 for content updated within 30 days, 71.8 for 31-90 days, 106.9 for 91-180 days, and 42.4 for 181-365 days. The 366+ day bucket had 67,973 items but no matched average because February GSC observations were unavailable for that group.

This makes staleness a usable directional signal for prioritizing human review, although it is not treated as a prediction of refresh success.

| Staleness bucket | n | Avg February impressions |
|---|---:|---:|
| 0-30 days | 34,076 | 796.5 |
| 31-90 days | 1,201 | 71.8 |
| 91-180 days | 18,239 | 106.9 |
| 181-365 days | 14,403 | 42.4 |
| 366+ days | 67,973 | N/A |

### Signal 2 — Search volume
**Verdict: MIXED**

Search volume is available as a demand-related signal and was discussed in the session in connection with quick-win opportunities. However, the measured relationship with February impressions is not monotonic. The 5,001+ search-volume bucket had 859.0 average February impressions, lower than several lower-volume buckets.

| Search-volume bucket | n | Avg February impressions |
|---|---:|---:|
| 0 | 163,631 | 1,305.2 |
| 1-100 | 174,178 | 1,303.5 |
| 101-500 | 24,316 | 1,432.4 |
| 501-1,000 | 5,890 | 1,319.6 |
| 1,001-5,000 | 6,274 | 1,159.8 |
| 5,001+ | 2,695 | 859.0 |

### Baseline rule

At the 2026-02-28 decision point, rank eligible content using a simple **staleness × demand** score:

**score = days_since_update × log(1 + search_volume)**

The rule gives more priority to content that has been unchanged for longer while still having measurable search demand. This is a directional review score, not a probability of refresh success.

**Reason code:** `STALE_DEMAND`

**Action label:** `REFRESH_REVIEW`

Rows updated after 2026-02-28 are excluded because they were not known at the decision cutoff.

In [15]:
q_baseline = f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        search_volume,
        DATE_DIFF(
            'day',
            content_updated_date,
            DATE '{DECISION_DATE}'
        ) AS days_since_update
    FROM read_parquet('{DIM_CONTENT_PATH}')
    WHERE content_updated_date IS NOT NULL
      AND content_updated_date <= DATE '{DECISION_DATE}'
),

scored AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        days_since_update,
        COALESCE(search_volume, 0) AS search_volume,

        days_since_update
        * LN(1 + COALESCE(search_volume, 0)) AS score

    FROM content
)

SELECT
    ROW_NUMBER() OVER (
        ORDER BY score DESC, client_hash_id, content_hash_id
    ) AS rank,

    client_hash_id,
    content_hash_id,
    content_updated_date,
    days_since_update,
    search_volume,
    ROUND(score, 4) AS score,

    'STALE_DEMAND' AS reason_code,
    'REFRESH_REVIEW' AS action

FROM scored
ORDER BY rank
"""

baseline_queue = con.execute(q_baseline).df()

display(baseline_queue.head(20))

,rank,client_hash_id,content_hash_id,content_updated_date,days_since_update,search_volume,score,reason_code,action
0,1,client_2e65897d94f60220,content_f3b2afe149271021,2025-07-03,240,8100,2159.9383,STALE_DEMAND,REFRESH_REVIEW
1,2,client_895724a3631e810c,content_96b0770e5deb2fbe,2025-08-09,203,33100,2112.6857,STALE_DEMAND,REFRESH_REVIEW
2,3,client_895724a3631e810c,content_b229e67c1c2db011,2025-08-09,203,33100,2112.6857,STALE_DEMAND,REFRESH_REVIEW
3,4,client_2e65897d94f60220,content_09d139e87bb9c6c6,2025-07-03,240,6600,2110.7943,STALE_DEMAND,REFRESH_REVIEW
4,5,client_895724a3631e810c,content_b50b506e2cf5974a,2025-08-11,201,33100,2091.8711,STALE_DEMAND,REFRESH_REVIEW
5,6,client_2e65897d94f60220,content_e4c5c2331bcb561e,2025-06-11,262,2900,2088.8764,STALE_DEMAND,REFRESH_REVIEW
6,7,client_2e65897d94f60220,content_6c027e4de1bc6839,2025-07-03,240,5400,2062.6415,STALE_DEMAND,REFRESH_REVIEW
7,8,client_2e65897d94f60220,content_fe04ae39b5c6ed9b,2025-06-10,263,2400,2047.0975,STALE_DEMAND,REFRESH_REVIEW
8,9,client_2e65897d94f60220,content_54077a01dc306ff9,2025-07-16,227,8100,2042.9416,STALE_DEMAND,REFRESH_REVIEW
9,10,client_2e65897d94f60220,content_b329a9f02802c4da,2025-07-16,227,8100,2042.9416,STALE_DEMAND,REFRESH_REVIEW


In [16]:
from pathlib import Path

OUTPUT_PATH = Path("../outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(baseline_queue):,} rows to:")
print(OUTPUT_PATH)

Wrote 135,892 rows to:
../outputs/baseline_action_score.csv


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
top20 = baseline_queue.head(20).copy()

display(
    top20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "content_updated_date",
            "days_since_update",
            "search_volume",
            "score",
            "reason_code",
            "action",
        ]
    ]
)

,rank,client_hash_id,content_hash_id,content_updated_date,days_since_update,search_volume,score,reason_code,action
0,1,client_2e65897d94f60220,content_f3b2afe149271021,2025-07-03,240,8100,2159.9383,STALE_DEMAND,REFRESH_REVIEW
1,2,client_895724a3631e810c,content_96b0770e5deb2fbe,2025-08-09,203,33100,2112.6857,STALE_DEMAND,REFRESH_REVIEW
2,3,client_895724a3631e810c,content_b229e67c1c2db011,2025-08-09,203,33100,2112.6857,STALE_DEMAND,REFRESH_REVIEW
3,4,client_2e65897d94f60220,content_09d139e87bb9c6c6,2025-07-03,240,6600,2110.7943,STALE_DEMAND,REFRESH_REVIEW
4,5,client_895724a3631e810c,content_b50b506e2cf5974a,2025-08-11,201,33100,2091.8711,STALE_DEMAND,REFRESH_REVIEW
5,6,client_2e65897d94f60220,content_e4c5c2331bcb561e,2025-06-11,262,2900,2088.8764,STALE_DEMAND,REFRESH_REVIEW
6,7,client_2e65897d94f60220,content_6c027e4de1bc6839,2025-07-03,240,5400,2062.6415,STALE_DEMAND,REFRESH_REVIEW
7,8,client_2e65897d94f60220,content_fe04ae39b5c6ed9b,2025-06-10,263,2400,2047.0975,STALE_DEMAND,REFRESH_REVIEW
8,9,client_2e65897d94f60220,content_54077a01dc306ff9,2025-07-16,227,8100,2042.9416,STALE_DEMAND,REFRESH_REVIEW
9,10,client_2e65897d94f60220,content_b329a9f02802c4da,2025-07-16,227,8100,2042.9416,STALE_DEMAND,REFRESH_REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

The queue ranks content using the `STALE_DEMAND` rule. Every item below is labeled `REFRESH_REVIEW`; the label means that the item should receive human review based on the baseline signals, not that a refresh is automatically justified.

| Rank | Action | Why it's here | What would make it wrong |
|---:|---|---|---|
| 1 | REFRESH_REVIEW | 240 days stale with 8,100 search volume, producing the highest score. | The content may already satisfy the search intent despite its age, or the demand signal may no longer reflect the current opportunity. |
| 2 | REFRESH_REVIEW | 203 days stale with 33,100 search volume. | High search demand may not translate into a refresh opportunity for this specific content item. |
| 3 | REFRESH_REVIEW | 203 days stale with 33,100 search volume. | The content could remain appropriate even though it is relatively old. |
| 4 | REFRESH_REVIEW | 240 days stale with 6,600 search volume. | Age alone may not indicate that the content requires editorial changes. |
| 5 | REFRESH_REVIEW | 201 days stale with 33,100 search volume. | Search volume may represent broader demand that is not directly addressable by this page. |
| 6 | REFRESH_REVIEW | 262 days stale with 2,900 search volume. | The page may be intentionally stable or already aligned with its intended search need. |
| 7 | REFRESH_REVIEW | 240 days stale with 5,400 search volume. | The observed signals do not show whether a refresh would improve the page. |
| 8 | REFRESH_REVIEW | 263 days stale with 2,400 search volume. | The page may not have a meaningful content gap despite being old. |
| 9 | REFRESH_REVIEW | 227 days stale with 8,100 search volume. | High demand does not establish that this page is the correct page to update. |
| 10 | REFRESH_REVIEW | 227 days stale with 8,100 search volume. | The content may still be performing adequately despite its age. |
| 11 | REFRESH_REVIEW | 227 days stale with 8,100 search volume. | The baseline lacks direct evidence that refreshing this particular item will help. |
| 12 | REFRESH_REVIEW | 227 days stale with 5,400 search volume. | Search demand may not indicate an actionable content problem. |
| 13 | REFRESH_REVIEW | 227 days stale with 5,400 search volume. | The page may already cover the relevant intent sufficiently. |
| 14 | REFRESH_REVIEW | 227 days stale with 5,400 search volume. | A human review could determine that no update is warranted. |
| 15 | REFRESH_REVIEW | 240 days stale with 2,900 search volume. | Staleness and demand alone cannot identify whether the content actually needs revision. |
| 16 | REFRESH_REVIEW | 227 days stale with 4,400 search volume. | The page could be intentionally maintained without a refresh. |
| 17 | REFRESH_REVIEW | 226 days stale with 4,400 search volume. | The available baseline signals do not measure content quality or search-intent fit. |
| 18 | REFRESH_REVIEW | 240 days stale with 2,400 search volume. | The page may not have an actionable opportunity even with measurable demand. |
| 19 | REFRESH_REVIEW | 203 days stale with 9,900 search volume. | Search volume alone cannot establish that a refresh would improve the page. |
| 20 | REFRESH_REVIEW | 203 days stale with 9,900 search volume. | The content may already be appropriate for its intended query or audience. |

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

### Weak picks

A few properties of the Top-20 show where the baseline can make weak recommendations:

- Several rows share identical staleness and search-volume values, so the ranking is not necessarily meaningful between tied candidates.
- High search volume does not consistently correspond to higher February impressions in the signal audit.
- The rule does not observe content quality, search-intent fit, ranking position, or whether a refresh would actually improve performance.
- Therefore, the Top-20 should be treated as a human-review queue rather than an automatic refresh decision.

### Leakage check

The baseline uses:

- `content_updated_date` relative to the fixed **2026-02-28** decision cutoff.
- `search_volume` from the content dimension.
- No March performance.
- No future outcome or label.
- No `march_had_impressions` field.
- No product/FlyRank flags as model inputs.
- No future-window performance features.

Rows with `content_updated_date > 2026-02-28` were explicitly excluded from the ranked queue.

The February impression analysis was used only to audit the signals and was not included in the baseline scoring formula.

**Conclusion:** the baseline is a decision-support ranking rule based on observable signals available at the stated cutoff, with human review required before taking an editorial action.

In [19]:
print("Rows:", len(baseline_queue))
print("Columns:", list(baseline_queue.columns))

print(
    "Post-cutoff rows:",
    (baseline_queue["content_updated_date"] > DECISION_DATE).sum()
)

print(
    "Missing scores:",
    baseline_queue["score"].isna().sum()
)

print(
    "Missing actions:",
    baseline_queue["action"].isna().sum()
)

display(baseline_queue.head(20))

Rows: 135892
Columns: ['rank', 'client_hash_id', 'content_hash_id', 'content_updated_date', 'days_since_update', 'search_volume', 'score', 'reason_code', 'action']
Post-cutoff rows: 0
Missing scores: 0
Missing actions: 0


,rank,client_hash_id,content_hash_id,content_updated_date,days_since_update,search_volume,score,reason_code,action
0,1,client_2e65897d94f60220,content_f3b2afe149271021,2025-07-03,240,8100,2159.9383,STALE_DEMAND,REFRESH_REVIEW
1,2,client_895724a3631e810c,content_96b0770e5deb2fbe,2025-08-09,203,33100,2112.6857,STALE_DEMAND,REFRESH_REVIEW
2,3,client_895724a3631e810c,content_b229e67c1c2db011,2025-08-09,203,33100,2112.6857,STALE_DEMAND,REFRESH_REVIEW
3,4,client_2e65897d94f60220,content_09d139e87bb9c6c6,2025-07-03,240,6600,2110.7943,STALE_DEMAND,REFRESH_REVIEW
4,5,client_895724a3631e810c,content_b50b506e2cf5974a,2025-08-11,201,33100,2091.8711,STALE_DEMAND,REFRESH_REVIEW
5,6,client_2e65897d94f60220,content_e4c5c2331bcb561e,2025-06-11,262,2900,2088.8764,STALE_DEMAND,REFRESH_REVIEW
6,7,client_2e65897d94f60220,content_6c027e4de1bc6839,2025-07-03,240,5400,2062.6415,STALE_DEMAND,REFRESH_REVIEW
7,8,client_2e65897d94f60220,content_fe04ae39b5c6ed9b,2025-06-10,263,2400,2047.0975,STALE_DEMAND,REFRESH_REVIEW
8,9,client_2e65897d94f60220,content_54077a01dc306ff9,2025-07-16,227,8100,2042.9416,STALE_DEMAND,REFRESH_REVIEW
9,10,client_2e65897d94f60220,content_b329a9f02802c4da,2025-07-16,227,8100,2042.9416,STALE_DEMAND,REFRESH_REVIEW


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.